# Data Mining – Lab 4
## Data Preparation: Data Quality, Cleaning, Integration, Transformation

**Dataset:** StudentsPerformance.csv  

**Лекцийн сэдэв:** Өгөгдөл бэлтгэх ба чанар (номын 2-р бүлэг)


### Lab зорилго
- Өгөгдлийн чанарыг (quality) шалгаж сурах
- Missing values болон noisy data-г цэвэрлэх аргуудыг практикт хэрэглэх
- Өгөгдөл нэгтгэх (integration), давхардал ба хамаарлыг (redundancy) илрүүлэх
- Normalization (min-max, z-score, decimal scaling) ба discretization хийх
- Lab 3-т үзсэн feature engineering-ийг бүрэн дүүрэн data preparation процесс болгон өргөтгөх

## 0) Dataset унших ба ерөнхий шалгалт

Өмнөх lab-уудын адил **StudentsPerformance.csv** dataset-ийг ашиглана.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group D,associate's degree,standard,completed,93,97,100
1,male,group B,master's degree,standard,none,72,51,63
2,female,group D,high school,standard,completed,53,66,64
3,female,group C,master's degree,standard,none,68,100,89
4,female,group D,bachelor's degree,standard,completed,100,100,100


In [2]:
print(df.shape)
df.info()

(1000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


## 1) Өгөгдлийн чанар (Data Quality)

Өгөгдөл нь ашиглах зорилгодоо нийцэж байвал чанартай гэж үзнэ.  
Чанарын хүчин зүйлс: **accuracy, completeness, consistency, timeliness, believability, interpretability**.


### TASK 1: Quality check

1. Багана бүрийн дутуу утгын тоог ол (completeness).
2. Давхардсан мөрийн тоог ол.
3. Оноонууд 0–100 мужид байгаа эсэхийг шалга (accuracy).
4. Категори багануудын өвөрмөц утгуудыг харж, бичих хэлбэр зөрүүтэй эсэхийг шалга (consistency).


In [3]:
# TODO: өгөгдлийн чанарын шалгалт
missing_per_col = df.isnull().sum()
print("Дутуу утга:\n", missing_per_col)

dup_count = df.duplicated().sum()
print("Давхардсан мөр:", dup_count)

score_cols = ['math score', 'reading score', 'writing score']
print(df[score_cols].agg(['min', 'max']))

for col in ['gender', 'lunch', 'test preparation course']:
    print(col, df[col].value_counts())


Дутуу утга:
 gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64
Давхардсан мөр: 0
     math score  reading score  writing score
min           0             17             10
max         100            100            100
gender gender
male      514
female    486
Name: count, dtype: int64
lunch lunch
standard        675
free/reduced    325
Name: count, dtype: int64
test preparation course test preparation course
none         610
completed    390
Name: count, dtype: int64


## 2) Data Cleaning – Missing values

StudentsPerformance.csv нь цэвэр dataset тул бодит нөхцөлийг дууриахын тулд  
зарим утгыг **зориудаар дутуу** болгоно (math, reading score-ийн 10%, lunch-ийн 5%).


In [4]:
rng = np.random.default_rng(42)
df_miss = df.copy()

for col in ['math score', 'reading score']:
    df_miss[col] = df_miss[col].astype(float)
    idx = rng.choice(df_miss.index, size=int(0.10 * len(df_miss)), replace=False)
    df_miss.loc[idx, col] = np.nan

idx = rng.choice(df_miss.index, size=int(0.05 * len(df_miss)), replace=False)
df_miss.loc[idx, 'lunch'] = np.nan

df_miss.isnull().sum()

gender                           0
race/ethnicity                   0
parental level of education      0
lunch                           50
test preparation course          0
math score                     100
reading score                  100
writing score                    0
dtype: int64

### TASK 2: Дутуу утгын хувь

Багана бүрийн дутуу утгын **хувийг** ол.


In [21]:
# TODO: дутуу утгын хувь
missing_pct = df_miss.isnull().sum() / len(df_miss) * 100
missing_pct

gender                          0.0
race/ethnicity                  0.0
parental level of education     0.0
lunch                           5.0
test preparation course         0.0
math score                     10.0
reading score                  10.0
writing score                   0.0
dtype: float64

### TASK 3: Missing values нөхөх аргууд

Дараах 5 аргыг хэрэглэ:

- a) Дутуу утгатай мөрийг орхих (ignore tuple)
- b) Баганын **mean**-ээр нөхөх
- c) Баганын **median**-аар нөхөх
- d) Ижил **class**-ийн mean-ээр нөхөх (gender-ээр бүлэглэнэ)
- e) Категори баганыг хамгийн их тохиолдох утгаар (**mode**) нөхөх


In [6]:
# TODO: missing values нөхөх
# a) Tuple-ийг орхих
df_a = df_miss.dropna()
print("a) үлдсэн мөр:", len(df_a))

# b) Mean
df_b = df_miss.copy()
df_b['math score'] = df_b['math score'].fillna(df_b['math score'].mean())

# c) Median
df_c = df_miss.copy()
df_c['reading score'] = df_c['reading score'].fillna(df_c['reading score'].median())

# d) Ижил class-ийн mean (gender-ээр)
df_d = df_miss.copy()
df_d['math score'] = df_d['math score'].fillna(
    df_d.groupby('gender')['math score'].transform('mean')
)

# e) Mode
df_e = df_miss.copy()
df_e['lunch'] = df_e['lunch'].fillna(df_e['lunch'].mode()[0])

print(df_b['math score'].isnull().sum(),
      df_d['math score'].isnull().sum(),
      df_e['lunch'].isnull().sum())

a) үлдсэн мөр: 772
0 0 0


### TASK 4: Аргуудыг харьцуулах

Анхны (дутуу утгагүй) `math score`-ын mean, std-ийг нөхсөн аргуудтай харьцуул.


In [7]:
# TODO: mean, std хэрхэн өөрчлөгдсөнийг харьцуул
comparison = pd.DataFrame({
    'original': df['math score'].agg(['mean', 'std']),
    'ignore_tuple': df_a['math score'].agg(['mean', 'std']),
    'mean_fill': df_b['math score'].agg(['mean', 'std']),
    'class_mean_fill': df_d['math score'].agg(['mean', 'std'])
})
comparison

,original,ignore_tuple,mean_fill,class_mean_fill
mean,70.105000,70.150259,70.212222,70.236315
std,14.468009,14.304842,13.481774,13.495603


### TASK 4.1: Тайлбар (үгээр)

1. Аль арга std-ийг хамгийн ихээр багасгав? Яагаад? 
Бүх утгыг gender тус бүрийн бус нийт оюутнуудын дунджаар сольсон mean_fill хамгийн их dataset-ийн утгыг хазайлгасан. Энэ нь үнэн утгаас холдуулсан гэж үзэж байна. Бусад орлуулга нь үнэнд илүү ойр сайн арга байсан тул original утгад арай ойр байна гэж дүгнэлээ.
2. Mean-ээр нөхөх нь өгөгдлийг хэрхэн хазайлгаж болох вэ? 
Уул нь mean утгаар орлуулах нь хазайлтад аль болох багаар нөлөөлөхийн тулд орлуулж байгаа арга юм. Хэрвээ бүх утга хоосон бус байсан dataset-ийн mean-ээр албаар хоосон болгосон утгуудыг нөхсөн бол хазайлт өөрчлөгдөхгүй байх байсан. Учир нь хазайлтыг тооцохдоо mean-ээс тухайн утгуудыг хасаж нийлбэрийг нь тооцдог. Тиймээс тэр зөрүүнүүд нь 0 болж хазайлт өөрчлөгдөх ёсгүй. Гэвч бид албаар хоосолсон утгуудыг тооцолгүй, үлдсэн утгуудын дундаж утгыг олсон тул энэ нь энэхүү зөрүүг үүсгэж байгаа гэж дүгнэж байна. Тиймээс дундаж утгаар орлуулах арга нь original dataset-ийн хазайлтаас хазайлгаж байна. Gender тус бүрийн дундаж нь original dataset-ийн утгуудтай илүү ойртож байгаа тул original datatset-ийн хазайлтаас арай бага хазайлттай байна.
3. Хэзээ "tuple орхих" арга зохимжтой вэ? 
Хэрвээ хоосон утгууд агуулсан мөрүүд нь бага байвал хэрэглэхэд тохиромжтой.


## 3) Data Cleaning – Noisy data

Noise нь хэмжсэн утга дахь санамсаргүй алдаа эсвэл хэлбэлзэл юм.  
Зөөллөх аргууд: **binning**, **regression**, **outlier analysis**.


### TASK 5: Binning – зөөллөх

Лекцийн жишээ: эрэмбэлсэн үнэ `4, 8, 15, 21, 21, 24, 25, 28, 34`.

1. Өгөгдлийг 3 bin-д (bin бүрт 3 утга) хуваа (equal-frequency).
2. **Bin means**-ээр зөөллө.
3. **Bin boundaries**-аар зөөллө.


In [8]:
prices = pd.Series([4, 8, 15, 21, 21, 24, 25, 28, 34])

# TODO: 3 bin-д хуваа
bins = [prices[i:i+3] for i in range(0, len(prices), 3)]

# Smoothing by bin means
smooth_mean = [[round(b.mean())] * len(b) for b in bins]

# Smoothing by bin boundaries
smooth_bound = []
for b in bins:
    lo, hi = b.min(), b.max()
    smooth_bound.append([int(lo) if (v - lo) <= (hi - v) else int(hi) for v in b])

print("Bins      :", [list(b) for b in bins])
print("By means  :", smooth_mean)
print("By bounds :", smooth_bound)

Bins      : [[4, 8, 15], [21, 21, 24], [25, 28, 34]]
By means  : [[9, 9, 9], [22, 22, 22], [29, 29, 29]]
By bounds : [[4, 4, 15], [21, 21, 24], [25, 25, 34]]


### TASK 6: Outlier илрүүлэх – IQR арга

`math score` дээр IQR аргаар outlier ол.

- Q1 = 25-р percentile, Q3 = 75-р percentile  
- IQR = Q3 − Q1  
- Outlier: `< Q1 − 1.5·IQR` эсвэл `> Q3 + 1.5·IQR`


In [9]:
# TODO: IQR outlier
q1 = df['math score'].quantile(0.25)
q3 = df['math score'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df['math score'] < lower) | (df['math score'] > upper)]
print("lower:", lower, "upper:", upper, "outlier тоо:", len(outliers))
outliers[['math score']].head()

lower: 30.0 upper: 110.0 outlier тоо: 8


,math score
59,0
76,8
211,18
316,22
338,23


### TASK 7: Outlier илрүүлэх – z-score арга

Mean-аас **2σ**-аас хол утгыг сэжигтэй outlier гэж үзье.


In [10]:
# TODO: z-score outlier
z = (df['math score'] - df['math score'].mean()) / df['math score'].std()
z_outliers = df[z.abs() > 2]
print("z-score outlier тоо:", len(z_outliers))

z-score outlier тоо: 43


### TASK 7.1: Тайлбар (үгээр)

1. IQR болон z-score аргаар олдсон outlier-ийн тоо ялгаатай байна уу? Яагаад?
IQR аргаар 8 outlier утга байсан бол z-score аргаар 43 outlier тогтоогдсон. Z-score аргад mean-ээс 2σ-аас хол байгаа утгуудыг outlier гэж үзсэн. The Empirical Rule (68-95-99.7 Rule) гэж байдаг. Уг нь mean-ээс зөрүү нь 2σ дотор байна гэдэг нь normal тархалтад нийт олонлогийн 95%-д багтах ёстой гэсэн дүрэм байдаг. Тиймээс outlier утгад олонлогийн outlier бус 5% орсон гэж дүгнэж болно. Тиймээс хоёр аргуудын утга ийм зөрүүтэй гарсан.
2. Оноо 0 байх нь "алдаа" юм уу, эсвэл бодит утга уу? Outlier-ийг устгахаас өмнө юуг шалгах ёстой вэ?
Оноо 0 байх нь алдаа биш. Шалгалтад 0 оноо авсан байх боломжтой. Хасах утга авах харин боломжгүй. Алдаа гэдэг нь боломжгүй утгыг хэлдэг байх ёстой. Outlier-ийг устгахаас өмнө зөвшөөрөгдөх хязгаарыг дахин бодож үзэх хэрэгтэй гэж бодож байна.


## 4) Data Integration – Өгөгдөл нэгтгэх

Олон эх сурвалжийн өгөгдлийг нэг санд нэгтгэхэд **entity identification**, **redundancy**,  
**tuple duplication**, **data value conflict** асуудлууд гарна.  

Бодит нөхцөлийг дууриахын тулд dataset-ийг хоёр хүснэгтэд хувааж, давхардал нэмнэ.


In [11]:
students = df[['gender', 'race/ethnicity', 'parental level of education',
               'lunch', 'test preparation course']].copy()
students.insert(0, 'student_id', range(1, len(df) + 1))

scores = df[['math score', 'reading score', 'writing score']].copy()
scores.insert(0, 'student_id', range(1, len(df) + 1))

# Давхардсан 5 мөр нэмнэ
scores_dup = pd.concat([scores, scores.sample(5, random_state=1)], ignore_index=True)
print(len(scores), "->", len(scores_dup))

1000 -> 1005


### TASK 8: Давхардал устгах ба нэгтгэх

1. `scores_dup` дахь давхардсан `student_id`-тай мөрийг устга.
2. `students` ба `scores_clean` хүснэгтийг `student_id` дээр нэгтгэ (inner join).


In [12]:
# TODO: duplicate устгаж, merge хий
scores_clean = scores_dup.drop_duplicates(subset=['student_id'])
merged = students.merge(scores_clean, on='student_id', how='inner')

print(len(scores_clean), merged.shape)
merged.head()

1000 (1000, 9)


,student_id,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,1,female,group D,associate's degree,standard,completed,93,97,100
1,2,male,group B,master's degree,standard,none,72,51,63
2,3,female,group D,high school,standard,completed,53,66,64
3,4,female,group C,master's degree,standard,none,68,100,89
4,5,female,group D,bachelor's degree,standard,completed,100,100,100


### TASK 9: Redundancy – Correlation analysis

Тоон attribute-уудын хоорондын корреляцийг ол. Өндөр корреляцтай attribute нь илүүдэл (redundant) байж болно.


In [13]:
# TODO: correlation matrix
corr = merged[['math score', 'reading score', 'writing score']].corr()
corr

,math score,reading score,writing score
math score,1.000000,0.735258,0.732973
reading score,0.735258,1.000000,0.819321
writing score,0.732973,0.819321,1.000000


### TASK 10: Nominal attribute – Chi-square тест

`gender` ба `lunch` хоёр хамааралтай эсэхийг χ² тестээр шалга.  
(p-value < 0.05 бол хамааралтай гэж үзнэ.)


In [15]:
from scipy.stats import chi2_contingency

# TODO: contingency table үүсгэ
contingency = pd.crosstab(merged['gender'], merged['lunch'])
chi2, p, dof, expected = chi2_contingency(contingency)

print("chi2 =", round(chi2, 3), " p-value =", round(p, 4), " dof =", dof)
contingency

chi2 = 1.63  p-value = 0.2018  dof = 1


lunch,free/reduced,standard
gender,,
female,148,338
male,177,337


### TASK 10.1: Тайлбар (үгээр)

1. Аль хоёр оноо хамгийн өндөр корреляцтай вэ? Тэдгээрийн нэгийг хасаж болох уу?
Reading болон writing score-ууд хамгийн хамааралтай байна. Магадгүй ижил чадвар шаарддаг чадвар тул ийм байгаа болов уу? Хамааралтай feature-уудыг аль болох хамт оруулах шаардлагагүй байдаг тул хасаж болно гэж бодож байна.
2. Chi-square тестийн үр дүнгээс юу дүгнэж байна вэ? 
р утга 0.05-аас их байгаа тул gender болон lunch feature-ууд хамааралгүй гэж дүгнэж болно.


## 5) Data Transformation – Normalization

Хэмжих нэгж, муж өөр attribute-ууд шинжилгээнд өөр жин авдаг.  
Normalization нь бүх attribute-д ижил жин өгөхийг зорино.


### TASK 11: Гурван normalization арга

`math score` дээр дараах аргуудыг хэрэглэ:

- **Min-Max:** `(v − min) / (max − min)`
- **Z-score:** `(v − mean) / std`
- **Decimal scaling:** `v / 10^j` (j нь max(|v'|) < 1 байх хамгийн бага бүхэл тоо)


In [16]:
# TODO: normalization
x = df['math score']

df['math_minmax'] = (x - x.min()) / (x.max() - x.min())
df['math_z'] = (x - x.mean()) / x.std()

j = len(str(int(x.abs().max())))
df['math_decimal'] = x / 10 ** j

df[['math score', 'math_minmax', 'math_z', 'math_decimal']].head()

,math score,math_minmax,math_z,math_decimal
0,93,0.93,1.582457,0.093
1,72,0.72,0.130979,0.072
2,53,0.53,-1.182264,0.053
3,68,0.68,-0.145493,0.068
4,100,1.00,2.066283,0.100


### TASK 12: Харьцуулалт

Дөрвөн баганын min, max, mean, std утгыг нэг хүснэгтээр харуул.


In [17]:
# TODO: харьцуулалтын хүснэгт
cols = ['math score', 'math_minmax', 'math_z', 'math_decimal']
df[cols].describe().loc[['min', 'max', 'mean', 'std']]

,math score,math_minmax,math_z,math_decimal
min,0.000000,0.00000,-4.845518e+00,0.000000
max,100.000000,1.00000,2.066283e+00,0.100000
mean,70.105000,0.70105,-2.859935e-16,0.070105
std,14.468009,0.14468,1.000000e+00,0.014468


### TASK 12.1: Тайлбар (үгээр)

1. Аргуудын гаралтын муж хэд вэ? Тус бүрийн min болон max утга нь гаралтын муж. minmax аргад 0-ээс 1 хооронд. zscore аргад хамгийн хол outlier утгууд mean-ээс хэдэн сигма зайд байгаа утга. math_decimal нь 0.0-ээс 0.1 хүртэл байна.
2. Outlier олонтой үед аль арга илүү тохиромжтой вэ? Яагаад? 
Zscore гэж бодож байна. Учир нь хэр хол сигмагаар зөрж байгаа утга нь шууд тооцогддог. Хэрэв абсолют утга нь 2-оос их байвал өмнөх шиг outlier утгуудыг олж шүүж болно.
3. Шинэ өгөгдөл ирэхэд яагаад min, max (эсвэл mean, std)-ийг хадгалах хэрэгтэй вэ? 
Магадгүй шинэ өгөгдлүүд өмнөх олонлогоос хэр утгуудыг тархааж, хазайлгаж буйг шинжиж болох юм.


## 6) Data Transformation – Discretization

Тоон утгыг интервал эсвэл ойлголтын шошгоор (Low, Medium, High) солино.


### TASK 13: Equal-width

`math score`-ыг **ижил өргөнтэй** 3 интервалд хуваа.


In [18]:
# TODO: equal-width
df['math_width'] = pd.cut(df['math score'], bins=3, labels=['Low', 'Medium', 'High'])
df['math_width'].value_counts().sort_index()

math_width
Low         8
Medium    387
High      605
Name: count, dtype: int64

### TASK 14: Equal-frequency

`math score`-ыг **ойролцоо тооны утгатай** 3 бүлэгт хуваа.


In [19]:
# TODO: equal-frequency
df['math_freq'] = pd.qcut(df['math score'], q=3, labels=['Low', 'Medium', 'High'])
df['math_freq'].value_counts().sort_index()

math_freq
Low       348
Medium    344
High      308
Name: count, dtype: int64

### TASK 15: Concept hierarchy (өөрийн босго)

Lab 3-ын ангиллыг ашигла:  
`< 60` → Low, `60–79` → Medium, `≥ 80` → High


In [20]:
# TODO: өөрийн босгоор discretization
df['math_level'] = pd.cut(df['math score'], bins=[0, 60, 80, 101],
                          labels=['Low', 'Medium', 'High'], right=False)
df['math_level'].value_counts().sort_index()

math_level
Low       228
Medium    510
High      262
Name: count, dtype: int64

### TASK 15.1: Тайлбар (үгээр)

1. Equal-width ба equal-frequency аргын bucket бүрийн тоо ямар ялгаатай байна вэ? 
equal-frequency нь бүлэг бүрт ойролцоо тооны утгуудыг агуулж байгаа бол equal-width нь бүлэг бүрт харьцангуй ялгаатай утгуудыг агуулж байна. Энэ нь бага оноотой хүүхдүүд илүү бага байгаатай холбоотой. 
2. Аль арга outlier-т илүү мэдрэмтгий вэ? equal-width арга мэдрэмтгий байна гэж бодож байна. Учир нь Олонлогийн min болон max утгаас хамаарч бүлэг бүрийн хязгаарыг тодорхойлдог.
3. Бодит хэрэглээнд аль аргыг хэзээ сонгох вэ? Өөрийн босго тавьсан арга нь бодит хэрэглээнд илүү үр дүнтэй.

## 7) Бүх алхмын дараалал (Concept Check)

Өгөгдөл бэлтгэх процесс: **чанар шалгах → цэвэрлэх → нэгтгэх → хувиргах**.


### TASK 16: Pipeline thinking (Verbal)

1. Дээрх 4 алхмыг яагаад энэ дарааллаар хийдэг вэ?
Өмнөх алхам нь бүрэн хийгдэж байж дараагийн үйлдэл нь утгатай болно. Чанаргүй датаг цэвэрлээд нэмэргүй. Цэвэрлээгүй датаг нэгтгэвэл дараа нь илүү төвөгтэй гэх мэт.
2. Normalization-ийг missing values нөхөхөөс ӨМНӨ хийвэл ямар асуудал гарах вэ? 
Mean-ээр нөхөх арга ашиглах бол normalization хийсний дараа адилхан л утга гарах байсан гэж бодож байна. Mean-ээр нөхөөд normalization хийх, normalization хийгээд mean-ээр нөхөх хоёр үйлдэл эцэст нь ижил утга гаргана. Учир нь normalization хоорондын scale-ийг хадгалдаг.
3. Классификацийн lab-д орохоос өмнө өөрийн dataset дээр аль алхмуудыг заавал хийх вэ? 
Процессийн бүх 4 үйлдлийг хийх нь зөв.


### Тайлбар
- 


# Data Mining – Lab 4 Тайлан
## Өгөгдөл бэлтгэх ба чанар

**Нэр:Цэлмэг**  
**Оюутны код: s22c031b**  
**Анги / Бүлэг: КУ-5**  
**Огноо:2026/10/26**  

---

## 1. Lab зорилго (1–2 өгүүлбэр)
Энэ лабораторийн ажлын зорилгыг өөрийн үгээр тайлбарлана уу.
Өгөгдлөө цэвэрлэн, эмхэтгэж classification хийхэд бэлтгэх.
---

## 2. Dataset-ийн товч тайлбар
- Dataset нэр: StudentsPerformance
- Мөр, баганын тоо: 1000 мөр, 8 багана
- Ашигласан үндсэн баганууд: Score баганууд болон gender, lunch category баганууд

---

## 3. Data Quality ба Cleaning

### 3.1 Missing values
- Ямар аргуудыг хэрэглэсэн бэ? 
(ignore, mean, median, class mean, mode)
- Аль нь хамгийн тохиромжтой байсан бэ? Яагаад?
ignore хийхэд асуудалгүй санагдсан, учир нь үлдсэн дата хангалттай олон мөртэй.

### 3.2 Noisy data
- Binning-ийн үр дүн (bin means / bin boundaries) юу байсан бэ? 
Noisy data-г зөөллөх зорилготой. Хоёр аргын bin boundaries нь тухайн bin-ий утга их утгадаа ойр байвал их утгыг нь багадаа ойр бол бага утгыг нь оноодог, bin mean нь бүх утгадаа bin-ий дундаж утгыг оноодог. Тус бүрийн хэрэглээ байгаа гэж бодож байна.
- IQR ба z-score аргаар хэдэн outlier олсон бэ? 8 болон 43. Normal distribution-ий тархалтын дүрмээр 2 сигма зайд 95%-ийн утгууд багтдаг тул zscore арга илүү их утгыг outlier гэж үзсэн.

---

## 4. Data Integration
- Хоёр хүснэгтийг яаж нэгтгэсэн бэ? student-id давхардахааргүйгээр нийлүүлсэн.
- Хэдэн давхардсан мөр устгасан бэ? 5.
- Correlation болон chi-square тестийн дүгнэлт: Writing болон Reading score-ууд хамгийн их хамааралтай бөгөөд lunch болон gender categorical feature-ууд хамааралгүй гэсэн дүгнэлтэд хүрсэн.

---

## 5. Data Transformation

### 5.1 Normalization
- Min-Max, z-score, decimal scaling аргуудын ялгаа юу вэ? Гаралтын мужийн утгаараа ялгаатай
- Аль аргыг хэзээ сонгох вэ? 
Outlier ихтэй үед Zscore, Min max утгууд хэд дахин их бус ойр зөрүүтэй бол decimal scaling, харин min-max ямар ч үед амар санагдсан.

### 5.2 Discretization
- Equal-width ба equal-frequency аргын үр дүнг харьцуулна уу.
Ижил өргөнтэй арга нь outlier утгатай үед тохиромжгүй. Бүлэг бүрийн утгын тоо харьцангуй гарна. Ижил давтамжтай арга нь бүлэг бүрт ямар ч үед ойролцоо тооны утгуудыг агуулна.
---

## 6. Сэтгэхүй
Data preparation-ийн 4 алхмын дарааллыг яагаад чухал гэж үзэж байна вэ? (3–4 өгүүлбэр)
Өгөгдлийн чанар сул байвал юуны өмнө нэмж дата олох, илүү feature-үүд нэмэх ёстой. Тэрнээс нааш цэвэрлээд нэмэргүй. Цэвэрлээгүй дата-г нэгтгэвэл дараа нь цэвэрлэхэд илүү хэцүү болно. Нэгтгээгүй үед утга учиртайгаар утгуудыг хувиргалт хийх боломжгүй. Тиймээс өмнөх алхмаа бүтэн хийж байж дараагийн алхам нь утгатай болдог.
---

## 7. Reflection

1. Өгөгдлийн чанар муу байвал data mining-ийн үр дүнд юу болох вэ? GIGO буюу Garbage in Garbage Out гэж үг байдаг. Үр дүн хог гарнаа л гэсэн үг.
2. Энэ lab дараагийн classification lab-д яаж бэлтгэж өгсөн бэ? Өгөгдлийг сургалтад хэрэглэхэд бэлэн болгосон.
3. Ямар хэсэг хамгийн ойлгомжгүй байсан бэ? Target variable одоо хүртэл тодорхой биш байгаа.

(3–5 өгүүлбэр)

---

## 8. Дүгнэлт

Өгөгдөл бэлтгэх процессийг ойлгож авсан. Датаг цэвэрлэх буюу дутуу утгуудыг нөхөх, сэжигтэй утгуудыг хасах аргуудыг сурсан.